# Gradient Descent in Code
### Part 4 — Probability Distributions & Machine Learning Formative

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1FZSNE297Sj5516_uOnQgqqxMFsR7yC1h#scrollTo=7B8G14TTJzGS)

---

> **Goal:** Convert manual gradient descent calculations into Python — step by step — using **NumPy**, **Matplotlib**, and **SciPy**.  
> Every update to `m` (slope) and `b` (intercept) is written **explicitly**. No black-box optimisers for the descent itself.

| Library | Role |
|---------|------|
| `numpy` | Vectorised array maths |
| `matplotlib` | Convergence plots, surface map, fitted line |
| `scipy.optimize.minimize` | BFGS cross-validation of the final result |

## 1. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize   # used later for SciPy cross-validation

print("Libraries loaded successfully.")

## 2. Define the Dataset and Initialize Parameters

We use a small dataset of $(X, Y)$ pairs.  
Initial values for `m` and `b` are set to **0**, and we choose a **learning rate** $\alpha$ and a number of **iterations**.

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
X = np.array([1, 2, 3, 4, 5], dtype=float)
Y = np.array([2, 4, 5, 4, 5], dtype=float)
N = len(X)   # number of data points

# ── Starting parameters ───────────────────────────────────────────────────────
m = 0.0      # slope (initial guess)
b = 0.0      # intercept (initial guess)

# ── Hyper-parameters ──────────────────────────────────────────────────────────
learning_rate = 0.01   # α  — step size
iterations    = 1000   # number of gradient descent steps

print(f"Dataset  : X = {X}")
print(f"          Y = {Y}")
print(f"N = {N}  |  Initial m = {m}  |  Initial b = {b}")
print(f"Learning rate α = {learning_rate}  |  Iterations = {iterations}")

### Dataset Preview

Before any computation, let's see what the raw data looks like.  
Five $(x_i, y_i)$ pairs — the same data used in the Part 3 manual calculations.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Visualisation — Raw Dataset Scatter Plot
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title("Raw Dataset:  X vs Y", fontsize=13, fontweight="bold")
ax.set_xlabel("X", fontsize=11)
ax.set_ylabel("Y", fontsize=11)
ax.grid(True, alpha=0.3)

ax.scatter(X, Y, color="steelblue", s=140, zorder=5,
           edgecolors="white", linewidths=0.8, label="Data points $(x_i,\\ y_i)$")

for xi, yi in zip(X, Y):
    ax.annotate(f"({xi:.0f}, {yi:.0f})",
                xy=(xi, yi), xytext=(xi + 0.08, yi + 0.18),
                fontsize=10, color="steelblue")

ax.set_xlim(0.3, 6.0)

ax.set_ylim(0.8, 6.5)plt.show()

ax.legend(fontsize=10)plt.tight_layout()

## 3. Implement the Cost Function (Mean Squared Error)

$$E(m, b) = \frac{1}{N} \sum_{i=1}^{N} \bigl(y_i - (m x_i + b)\bigr)^2$$

A lower error means our line fits the data better.

In [ ]:
def compute_mse(X, Y, m, b):
    """Mean Squared Error: E = (1/N) * sum((y_i - (m*x_i + b))^2)"""
    errors = Y - (m * X + b)
    mse    = (1 / N) * np.sum(errors ** 2)
    return mse

# Sanity-check with initial parameters (m=0, b=0)
initial_error = compute_mse(X, Y, m, b)
print(f"Initial MSE (m=0, b=0) = {initial_error:.4f}")

## 4. Implement the Gradient Descent Update Step

The partial derivatives of the MSE with respect to `m` and `b` are:

$$\frac{\partial E}{\partial m} = \frac{-2}{N} \sum_{i=1}^{N} x_i \bigl(y_i - (m x_i + b)\bigr)$$

$$\frac{\partial E}{\partial b} = \frac{-2}{N} \sum_{i=1}^{N} \bigl(y_i - (m x_i + b)\bigr)$$

The update rules are:

$$m \leftarrow m - \alpha \frac{\partial E}{\partial m}, \qquad b \leftarrow b - \alpha \frac{\partial E}{\partial b}$$

Both gradients are computed **before** either parameter is updated (simultaneous update).

In [ ]:
def gradient_descent_step(X, Y, m, b, learning_rate):
    """
    Perform ONE gradient descent update for m and b.

    Step 1 – compute residuals:  e_i = y_i - (m*x_i + b)
    Step 2 – compute gradients:
         dE/dm = (-2/N) * sum( x_i * e_i )
         dE/db = (-2/N) * sum( e_i )
    Step 3 – update parameters simultaneously:
         m_new = m - α * dE/dm
         b_new = b - α * dE/db
    """
    # Step 1: residuals
    residuals = Y - (m * X + b)

    # Step 2: gradients (partial derivatives)
    dE_dm = (-2 / N) * np.sum(X * residuals)
    dE_db = (-2 / N) * np.sum(residuals)

    # Step 3: simultaneous parameter update
    m_new = m - learning_rate * dE_dm
    b_new = b - learning_rate * dE_db

    return m_new, b_new, dE_dm, dE_db

# Quick demonstration of a single step from m=0, b=0
m_test, b_test, g_m, g_b = gradient_descent_step(X, Y, 0.0, 0.0, learning_rate)
print(f"After 1 step  →  m = {m_test:.6f}  |  b = {b_test:.6f}")
print(f"Gradients     →  dE/dm = {g_m:.6f}  |  dE/db = {g_b:.6f}")

## 5. Run Gradient Descent and Track History

We loop for `iterations` steps, explicitly calling the update function each time and recording `m`, `b`, and the **error** at every iteration.

In [ ]:
# ── Re-initialise from scratch ────────────────────────────────────────────────
m = 0.0
b = 0.0

# ── History lists ─────────────────────────────────────────────────────────────
m_history     = []
b_history     = []
error_history = []

# ── Gradient Descent loop ─────────────────────────────────────────────────────
for i in range(iterations):

    # --- record current state BEFORE the update ---
    current_error = compute_mse(X, Y, m, b)
    m_history.append(m)
    b_history.append(b)
    error_history.append(current_error)

    # --- perform one explicit update step ---
    m, b, dE_dm, dE_db = gradient_descent_step(X, Y, m, b, learning_rate)

    # --- print every iteration so all intermediate results are visible ---
    print(f"Iter {i+1:4d}  |  m = {m:.6f}  |  b = {b:.6f}  |  "
          f"dE/dm = {dE_dm:+.6f}  |  dE/db = {dE_db:+.6f}  |  MSE = {current_error:.6f}")

### Error Surface & Gradient Descent Path

The MSE forms a **convex bowl** in $(m, b)$ space — there is exactly one global minimum.  
The white path shows the trajectory our algorithm walked across that surface iteration by iteration, from the red start point $(m=0, b=0)$ down to the green convergence point.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Visualisation — MSE Error Surface with Gradient Descent Path Overlay
# ══════════════════════════════════════════════════════════════════════════════
m_range = np.linspace(-0.2, 0.85, 300)
b_range = np.linspace(-0.2, 2.7, 300)
MM, BB  = np.meshgrid(m_range, b_range)
ZZ      = np.vectorize(lambda mi, bi: compute_mse(X, Y, mi, bi))(MM, BB)

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_title("MSE Error Surface  ·  Gradient Descent Path", fontsize=13, fontweight="bold")
ax.set_xlabel("m  (slope)", fontsize=11)
ax.set_ylabel("b  (intercept)", fontsize=11)
ax.grid(True, alpha=0.3)

# Filled colour-map + white iso-lines
contf = ax.contourf(MM, BB, ZZ, levels=60, cmap="RdYlGn_r", alpha=0.88)
plt.colorbar(contf, ax=ax, label="MSE", shrink=0.85)
ax.contour(MM, BB, ZZ, levels=25, colors="white", linewidths=0.35, alpha=0.35)

# GD path (downsampled to ≈80 points for readability)
step   = max(1, len(m_history) // 80)
gd_m   = m_history[::step] + [m_history[-1]]
gd_b   = b_history[::step] + [b_history[-1]]
ax.plot(gd_m, gd_b, color="white", linewidth=1.8, alpha=0.9, zorder=4)

# Start / end markers
ax.scatter([m_history[0]],  [b_history[0]],
           color="red",  s=130, zorder=6, label="Start  (m=0, b=0)")
ax.scatter([m_history[-1]], [b_history[-1]],
           color="lime", s=130, zorder=6,
           label=f"End  (m≈{m_history[-1]:.4f},  b≈{b_history[-1]:.4f})")

ax.legend(fontsize=10, framealpha=0.7)

plt.tight_layout()plt.show()

## 6. Compute Final Predictions

Using the optimised `m` and `b`, compute:

$$\hat{Y} = m X + b$$

We also cross-validate the result using **SciPy's `minimize`** on the same MSE objective.

In [ ]:
# ── Final parameters from our manual gradient descent ─────────────────────────
print("=" * 55)
print(f"  Final  m  (slope)     = {m:.6f}")
print(f"  Final  b  (intercept) = {b:.6f}")
print(f"  Final  MSE            = {compute_mse(X, Y, m, b):.6f}")
print("=" * 55)

# ── Predictions: Ŷ = m*X + b ────────────────────────────────────────────────
Y_hat = m * X + b

print("\n  x   |  Actual Y  |  Predicted Ŷ  |  Error (Y - Ŷ)")
print("  " + "-" * 50)
for xi, yi, yi_hat in zip(X, Y, Y_hat):
    print(f"  {xi:.0f}   |   {yi:.4f}   |    {yi_hat:.4f}     |  {yi - yi_hat:+.4f}")

# ── SciPy cross-validation ─────────────────────────────────────────────────────
def mse_scipy(params):
    """Objective function for SciPy: params = [m, b]"""
    return compute_mse(X, Y, params[0], params[1])

scipy_result  = minimize(mse_scipy, x0=[0.0, 0.0], method="BFGS")
m_scipy, b_scipy = scipy_result.x

print("\n── SciPy minimize (BFGS) cross-check ──")
print(f"  SciPy  m  = {m_scipy:.6f}  (manual GD: {m:.6f})")
print(f"  SciPy  b  = {b_scipy:.6f}  (manual GD: {b:.6f})")

### Fitted Regression Line & Residuals

The crimson line is the best-fit line $\hat{Y} = mX + b$ found by gradient descent.  
The dashed grey drop-lines show the **residual** $\varepsilon_i = y_i - \hat{y}_i$ for each data point — how far each actual value sits from the prediction.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Visualisation — Fitted Regression Line with Residuals
# ══════════════════════════════════════════════════════════════════════════════
X_dense = np.linspace(0.5, 5.5, 200)
Y_line  = m * X_dense + b

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(f"Fitted Line:  Ŷ = {m:.4f}·X + {b:.4f}", fontsize=13, fontweight="bold")
ax.set_xlabel("X", fontsize=11)
ax.set_ylabel("Y", fontsize=11)
ax.grid(True, alpha=0.3)

# Regression line
ax.plot(X_dense, Y_line, color="crimson", linewidth=2.2,
        label=f"Ŷ = {m:.4f}·X + {b:.4f}")

# Residual drop-lines and annotations
for xi, yi, yi_hat in zip(X, Y, Y_hat):
    ax.plot([xi, xi], [yi, yi_hat],
            color="gray", linestyle="--", linewidth=1.2, alpha=0.65)
    ax.annotate(f"ε = {yi - yi_hat:+.3f}",
                xy=(xi, (yi + yi_hat) / 2),
                xytext=(xi + 0.09, (yi + yi_hat) / 2),
                fontsize=8.5, color="dimgray")

# Actual data points (on top)
ax.scatter(X, Y, color="steelblue", s=120, zorder=5,
           edgecolors="white", linewidths=0.8, label="Actual Y")

ax.set_xlim(0.3, 6.2)
ax.legend(fontsize=10)

plt.tight_layout()plt.show()

## 7. Visualize Parameter and Error Changes Over Iterations

Two final convergence plots confirm the algorithm worked:

| Plot | What it shows |
|------|--------------|
| **Plot 1** | `m` and `b` rising from 0 toward their optimal values, then flattening |
| **Plot 2** | MSE dropping steeply from ≈ 17.5 (at m=0, b=0) and converging at ≈ 0.48 |

In [ ]:
iter_axis = np.arange(1, iterations + 1)

# ══════════════════════════════════════════════════════════════════════════════
# Plot 1 —  m and b convergence
# ══════════════════════════════════════════════════════════════════════════════
fig1, ax1 = plt.subplots(figsize=(10, 4))
ax1.set_title("Convergence of m and b over Gradient Descent Iterations", fontsize=13, fontweight="bold")
ax1.set_xlabel("Iteration", fontsize=11)
ax1.set_ylabel("Parameter Value", fontsize=11)
ax1.grid(True, alpha=0.3)

ax1.plot(iter_axis, m_history, label="m  (slope)",     color="steelblue",  linewidth=2)
ax1.plot(iter_axis, b_history, label="b  (intercept)", color="darkorange", linewidth=2, linestyle="--")
ax1.axhline(y=m, color="steelblue",  linestyle=":", alpha=0.5, label=f"m converged ≈ {m:.4f}")
ax1.axhline(y=b, color="darkorange", linestyle=":", alpha=0.5, label=f"b converged ≈ {b:.4f}")
ax1.legend(fontsize=10)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.set_title("Mean Squared Error (MSE) over Gradient Descent Iterations", fontsize=13, fontweight="bold")
ax2.set_xlabel("Iteration", fontsize=11)
ax2.set_ylabel("MSE", fontsize=11)
ax2.grid(True, alpha=0.3)

ax2.plot(iter_axis, error_history, color="crimson", linewidth=2, label="MSE (Training Error)")
ax2.axhline(y=error_history[-1], color="crimson", linestyle=":", alpha=0.5,
            label=f"Final MSE ≈ {error_history[-1]:.4f}")
ax2.legend(fontsize=10)

plt.tight_layout()
plt.tight_layout()plt.show()
plt.show()